In [11]:
#!/usr/bin/env python3
"""
YOLO Training Script - Optimized for H. pylori Detection

Carefully tuned for medical imaging:
- Conservative augmentation (no crazy transforms)
- Proper validation strategy
- Early stopping to prevent overfitting
- Medical imaging best practices

Based on your consensus dataset:
- 232 patches (high confidence)
- ~300 bacteria objects
- Split: 186 train / 46 val
"""

import os
from pathlib import Path
import yaml
import json
import shutil
from datetime import datetime

# ====================================================================
# CONFIGURATION - OPTIMIZED FOR YOUR DATASET
# ====================================================================

# Paths
CONSENSUS_JSON = "/Users/user/py/Hpylori/consensus_patches.json"
SOURCE_PATCHES = "/Users/user/py/Hpylori/patches2"  # Classifier 3 (has all consensus)
SOURCE_LABELS = "/Users/user/py/Hpylori/labels2"
OUTPUT_DIR = "/Users/user/py/Hpylori/yolo_training"

# Model selection
MODEL = "yolov8n.pt"  # Nano - fast, good for small datasets
# Alternatives:
# "yolov8s.pt"  # Small - more accurate, slightly slower
# "yolov8m.pt"  # Medium - best accuracy, needs more data

# Training parameters - OPTIMIZED
EPOCHS = 50              # Enough to converge, not too many
PATIENCE = 30             # Early stopping (stop if no improvement for 30 epochs)
IMG_SIZE = 640              # Larger input
LEARNING_RATE = 0.0001      # More conservative
BATCH_SIZE = 8              # More updates per epoch
PATIENCE = 40               # More patience

# Add to augmentation config:
AUGMENTATION_CONFIG = {
    'hsv_h': 0.01,
    'hsv_s': 0.3,
    'hsv_v': 0.2,
    'degrees': 90,
    'translate': 0.1,
    'scale': 0.2,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'mosaic': 0.0,
    'mixup': 0.0,
    'copy_paste': 0.0,
    'conf': 0.001,           # ADD THIS - very low initial confidence threshold
}
# Data split
TRAIN_RATIO = 0.8         # 80% train, 20% val
RANDOM_SEED = 42

# Class weights (if imbalanced)
CLASS_WEIGHTS = None  # None for balanced, or [1.0] for single class

# ====================================================================
# STEP 1: LOAD CONSENSUS DATA
# ====================================================================

def load_consensus_patches():
    """Load consensus patch information"""
    print("="*70)
    print("LOADING CONSENSUS DATASET")
    print("="*70)
    
    with open(CONSENSUS_JSON, 'r') as f:
        data = json.load(f)
    
    high_conf_patches = data['high_confidence']
    
    print(f"\nConsensus patches: {len(high_conf_patches)}")
    print(f"  All 3 agree: {data['statistics']['all_3_agree_count']}")
    print(f"  Any 2 agree: {data['statistics']['any_2_agree_count']}")
    
    return high_conf_patches

# ====================================================================
# STEP 2: ORGANIZE DATASET
# ====================================================================

def organize_dataset(consensus_patches):
    """Create YOLO directory structure and copy files"""
    print("\n" + "="*70)
    print("ORGANIZING DATASET")
    print("="*70)
    
    from sklearn.model_selection import train_test_split
    
    output_dir = Path(OUTPUT_DIR)
    
    # Create directories
    dirs = {
        'train_images': output_dir / 'images' / 'train',
        'val_images': output_dir / 'images' / 'val',
        'train_labels': output_dir / 'labels' / 'train',
        'val_labels': output_dir / 'labels' / 'val'
    }
    
    for dir_path in dirs.values():
        dir_path.mkdir(parents=True, exist_ok=True)
    
    print("✓ Created directory structure")
    
    # Find patch files
    source_patches = Path(SOURCE_PATCHES)
    source_labels = Path(SOURCE_LABELS)
    
    patch_files = []
    missing = 0
    
    for patch_info in consensus_patches:
        x, y = patch_info['coord']
        
        # Find matching file
        pattern = f"*_x{x:05d}_y{y:05d}_*.png"
        matches = list(source_patches.glob(pattern))
        
        if not matches:
            missing += 1
            continue
        
        patch_file = matches[0]
        label_file = source_labels / (patch_file.stem + ".txt")
        
        if patch_file.exists() and label_file.exists():
            patch_files.append({
                'patch': patch_file,
                'label': label_file
            })
    
    print(f"\n✓ Found {len(patch_files)} valid patch-label pairs")
    if missing > 0:
        print(f"⚠ Missing {missing} files")
    
    # Split train/val
    train_files, val_files = train_test_split(
        patch_files,
        test_size=1-TRAIN_RATIO,
        random_state=RANDOM_SEED
    )
    
    print(f"\nDataset split:")
    print(f"  Training:   {len(train_files)} patches")
    print(f"  Validation: {len(val_files)} patches")
    
    # Count objects
    def count_objects(files):
        total = 0
        for f in files:
            with open(f['label'], 'r') as lf:
                total += len([l for l in lf.readlines() if l.strip()])
        return total
    
    train_objects = count_objects(train_files)
    val_objects = count_objects(val_files)
    
    print(f"\nObjects:")
    print(f"  Training:   {train_objects} bacteria")
    print(f"  Validation: {val_objects} bacteria")
    print(f"  Total:      {train_objects + val_objects} bacteria")
    
    # Copy files
    print("\nCopying files...")
    
    for file_info in train_files:
        shutil.copy2(file_info['patch'], dirs['train_images'])
        shutil.copy2(file_info['label'], dirs['train_labels'])
    
    for file_info in val_files:
        shutil.copy2(file_info['patch'], dirs['val_images'])
        shutil.copy2(file_info['label'], dirs['val_labels'])
    
    print("✓ Files copied")
    
    return len(train_files), len(val_files), train_objects, val_objects

# ====================================================================
# STEP 3: CREATE DATASET.YAML
# ====================================================================

def create_dataset_yaml():
    """Create YOLO configuration file"""
    print("\n" + "="*70)
    print("CREATING DATASET CONFIGURATION")
    print("="*70)
    
    config = {
        'path': str(Path(OUTPUT_DIR).absolute()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 1,
        'names': ['Bacteria']
    }
    
    yaml_path = Path(OUTPUT_DIR) / 'dataset.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✓ Created {yaml_path}")
    
    return yaml_path

# ====================================================================
# STEP 4: CREATE CUSTOM TRAINING CONFIG
# ====================================================================

def create_training_config():
    """Create custom hyperparameters for medical imaging"""
    print("\n" + "="*70)
    print("CREATING TRAINING CONFIGURATION")
    print("="*70)
    
    config_path = Path(OUTPUT_DIR) / 'hyp.yaml'
    
    hyperparameters = {
        'lr0': LEARNING_RATE,
        'lrf': 0.01,
        'momentum': 0.937,
        'weight_decay': 0.0005,
        'warmup_epochs': 3.0,
        'warmup_momentum': 0.8,
        'warmup_bias_lr': 0.1,
        'box': 7.5,
        'cls': 0.5,
        'dfl': 1.5,
        **AUGMENTATION_CONFIG
    }
    
    with open(config_path, 'w') as f:
        yaml.dump(hyperparameters, f, default_flow_style=False)
    
    print(f"✓ Created {config_path}")
    print("\nAugmentation settings (CONSERVATIVE):")
    print(f"  Rotation: ±{AUGMENTATION_CONFIG['degrees']}°")
    print(f"  Scale: ±{AUGMENTATION_CONFIG['scale']*100}%")
    print(f"  Translation: ±{AUGMENTATION_CONFIG['translate']*100}%")
    print(f"  Flips: Horizontal & Vertical")
    print(f"  Color: Minimal (HSV variation for staining)")
    print("\n✗ Disabled transforms:")
    print("  - Shear (unrealistic)")
    print("  - Perspective (microscope is flat)")
    print("  - Mosaic (confusing)")
    print("  - Mixup (confusing)")
    print("  - Copy-paste (context matters)")
    
    return config_path

# ====================================================================
# STEP 5: TRAIN MODEL
# ====================================================================

def train_model(yaml_path, hyp_path):
    """Train YOLO model with optimized settings"""
    print("\n" + "="*70)
    print("TRAINING YOLO MODEL")
    print("="*70)
    
    try:
        from ultralytics import YOLO
    except ImportError:
        print("\n❌ ultralytics not installed!")
        print("Install: pip install ultralytics")
        return None
    
    # Training settings
    print("\nModel configuration:")
    print(f"  Model: {MODEL}")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Image size: {IMG_SIZE}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"  Patience: {PATIENCE} (early stopping)")
    
    # Initialize model
    model = YOLO(MODEL)
    
    # Train
    print("\n" + "="*70)
    print("STARTING TRAINING...")
    print("="*70)
    print("\nThis will take 1-2 hours.")
    print("You can monitor progress in the terminal.\n")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        save=True,
        save_period=10,
        cache=False,
        device='mps',  # Use Apple Silicon GPU if available, else 'cpu' or '0' for CUDA
        workers=4,
        project=str(Path(OUTPUT_DIR)),
        name=f'train_{timestamp}',
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        verbose=True,
        seed=RANDOM_SEED,
        deterministic=True,
        single_cls=True,  # Single class detection
        rect=False,  # Don't use rectangular training
        cos_lr=True,  # Cosine learning rate scheduler
        close_mosaic=10,  # Disable mosaic in last 10 epochs
        amp=True,  # Automatic mixed precision
        fraction=1.0,  # Use 100% of data
        profile=False,
        # Load custom hyperparameters
        cfg=str(hyp_path)
    )
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    
    # Find best model
    train_dir = Path(OUTPUT_DIR) / f'train_{timestamp}'
    best_model = train_dir / 'weights' / 'best.pt'
    last_model = train_dir / 'weights' / 'last.pt'
    
    print(f"\nModel saved:")
    print(f"  Best:  {best_model}")
    print(f"  Last:  {last_model}")
    print(f"  Metrics: {train_dir / 'results.csv'}")
    print(f"  Plots: {train_dir}")
    
    return best_model, results

# ====================================================================
# STEP 6: EVALUATE MODEL
# ====================================================================

def evaluate_model(model_path):
    """Evaluate trained model"""
    print("\n" + "="*70)
    print("EVALUATING MODEL")
    print("="*70)
    
    try:
        from ultralytics import YOLO
    except ImportError:
        return
    
    if not model_path or not Path(model_path).exists():
        print("❌ Model not found")
        return
    
    model = YOLO(str(model_path))
    
    # Validate
    print("\nRunning validation...")
    metrics = model.val()
    
    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    
    print(f"\nmAP Scores:")
    print(f"  mAP50:     {metrics.box.map50:.3f}  (IoU=0.5)")
    print(f"  mAP50-95:  {metrics.box.map:.3f}  (IoU=0.5:0.95)")
    
    print(f"\nPer-Class Metrics:")
    print(f"  Precision: {metrics.box.mp:.3f}")
    print(f"  Recall:    {metrics.box.mr:.3f}")
    print(f"  F1 Score:  {2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr):.3f}")
    
    print("\n" + "="*70)
    print("INTERPRETATION")
    print("="*70)
    
    map50 = metrics.box.map50
    
    if map50 > 0.75:
        print("\n🎉 EXCELLENT! Model is performing very well!")
        print("  → Ready for production use")
        print("  → Consider testing on new slides")
    elif map50 > 0.65:
        print("\n✓ GOOD! Model is performing well.")
        print("  → Suitable for clinical use")
        print("  → Can be improved with more data")
    elif map50 > 0.50:
        print("\n✓ DECENT baseline achieved.")
        print("  → Needs improvement for production")
        print("  → Add more training data")
        print("  → Try active learning")
    else:
        print("\n⚠️ Model needs improvement.")
        print("  → Check annotations quality")
        print("  → Add more diverse training data")
        print("  → Verify image quality")
    
    return metrics

# ====================================================================
# STEP 7: GENERATE TRAINING REPORT
# ====================================================================

def generate_report(train_count, val_count, train_obj, val_obj, metrics, model_path):
    """Generate training summary report"""
    print("\n" + "="*70)
    print("TRAINING SUMMARY REPORT")
    print("="*70)
    
    report = {
        'timestamp': datetime.now().isoformat(),
        'dataset': {
            'total_patches': train_count + val_count,
            'train_patches': train_count,
            'val_patches': val_count,
            'total_objects': train_obj + val_obj,
            'train_objects': train_obj,
            'val_objects': val_obj,
            'source': 'Consensus dataset (2+ classifiers agree)'
        },
        'training': {
            'model': MODEL,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'image_size': IMG_SIZE,
            'learning_rate': LEARNING_RATE,
            'patience': PATIENCE
        },
        'augmentation': AUGMENTATION_CONFIG,
        'results': {
            'mAP50': float(metrics.box.map50),
            'mAP50-95': float(metrics.box.map),
            'precision': float(metrics.box.mp),
            'recall': float(metrics.box.mr),
            'f1_score': float(2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr))
        },
        'model_path': str(model_path)
    }
    
    report_path = Path(OUTPUT_DIR) / 'training_report.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"\n✓ Report saved: {report_path}")
    
    # Print summary
    print("\n📊 Quick Summary:")
    print(f"  Dataset: {train_count + val_count} consensus patches")
    print(f"  Objects: {train_obj + val_obj} bacteria")
    print(f"  mAP50: {metrics.box.map50:.1%}")
    print(f"  F1: {report['results']['f1_score']:.1%}")
    print(f"  Model: {model_path}")


In [10]:
# ====================================================================
# MAIN EXECUTION
# ====================================================================

print("\n" + "="*70)
print("YOLO TRAINING - OPTIMIZED FOR H. PYLORI DETECTION")
print("="*70)
    
print("\nDataset: Consensus patches (validated by 2+ classifiers)")
print("Strategy: Conservative augmentation + early stopping")
print("Goal: 65-75% mAP50 (excellent for first model)")
    
# Step 1: Load data
consensus_patches = load_consensus_patches()
    
# Step 2: Organize dataset
train_count, val_count, train_obj, val_obj = organize_dataset(consensus_patches)
    
# Step 3: Create configs
yaml_path = create_dataset_yaml()
hyp_path = create_training_config()
    


YOLO TRAINING - OPTIMIZED FOR H. PYLORI DETECTION

Dataset: Consensus patches (validated by 2+ classifiers)
Strategy: Conservative augmentation + early stopping
Goal: 65-75% mAP50 (excellent for first model)
LOADING CONSENSUS DATASET

Consensus patches: 232
  All 3 agree: 0
  Any 2 agree: 232

ORGANIZING DATASET
✓ Created directory structure

✓ Found 232 valid patch-label pairs

Dataset split:
  Training:   185 patches
  Validation: 47 patches

Objects:
  Training:   238 bacteria
  Validation: 64 bacteria
  Total:      302 bacteria

Copying files...
✓ Files copied

CREATING DATASET CONFIGURATION
✓ Created /Users/user/py/Hpylori/yolo_training/dataset.yaml

CREATING TRAINING CONFIGURATION
✓ Created /Users/user/py/Hpylori/yolo_training/hyp.yaml

Augmentation settings (CONSERVATIVE):
  Rotation: ±90°
  Scale: ±20.0%
  Translation: ±10.0%
  Flips: Horizontal & Vertical
  Color: Minimal (HSV variation for staining)

✗ Disabled transforms:
  - Shear (unrealistic)
  - Perspective (microscope 

In [12]:
# Check if labels match images
def verify_dataset():
    train_imgs = list(Path(OUTPUT_DIR, 'images/train').glob('*.png'))
    train_lbls = list(Path(OUTPUT_DIR, 'labels/train').glob('*.txt'))
    
    print(f"Training images: {len(train_imgs)}")
    print(f"Training labels: {len(train_lbls)}")
    
    # Check for empty labels
    empty = 0
    for lbl in train_lbls:
        if os.path.getsize(lbl) == 0:
            empty += 1
    
    if empty > 0:
        print(f"⚠️  {empty} empty label files!")
    
verify_dataset()

Training images: 185
Training labels: 185


In [13]:
 # Confirm before training
print("\n" + "="*70)
print("READY TO TRAIN")
print("="*70)
    
print("\nFinal configuration:")
print(f"  Training patches:   {train_count}")
print(f"  Validation patches: {val_count}")
print(f"  Total objects:      {train_obj + val_obj}")
print(f"  Estimated time:     1-2 hours")
    
# Step 4: Train
model_path, results = train_model(yaml_path, hyp_path)



READY TO TRAIN

Final configuration:
  Training patches:   185
  Validation patches: 47
  Total objects:      302
  Estimated time:     1-2 hours

TRAINING YOLO MODEL

Model configuration:
  Model: yolov8n.pt
  Epochs: 50
  Batch size: 8
  Image size: 640
  Learning rate: 0.0001
  Patience: 40 (early stopping)

STARTING TRAINING...

This will take 1-2 hours.
You can monitor progress in the terminal.

New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.223 🚀 Python-3.13.2 torch-2.6.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=/Users/user/py/Hpylori/yolo_training/hyp.yaml, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/user/py/Hpylori/yolo_training/dataset.yaml, degrees=90, deterministic=True, device=mps, dfl=1.5, dnn=False, dr

In [14]:
# Step 5: Evaluate
metrics = evaluate_model(model_path)


EVALUATING MODEL

Running validation...
Ultralytics 8.3.223 🚀 Python-3.13.2 torch-2.6.0 CPU (Apple M2)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 874.4±64.7 MB/s, size: 602.2 KB)
val: Scanning /Users/user/py/Hpylori/yolo_training/labels/val.cache... 47 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 47/47 171.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 0.3it/s 9.8s7.6ss
                   all         47         64      0.516      0.547      0.503      0.241
Speed: 1.0ms preprocess, 188.4ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /Users/user/py/Hpylori/runs/detect/val

RESULTS

mAP Scores:
  mAP50:     0.503  (IoU=0.5)
  mAP50-95:  0.241  (IoU=0.5:0.95)

Per-Class Metrics:
  Precision: 0.516
  Recall:    0.547
  F1 Score:  0.531

INTERPRETATION

✓ DECENT baseline achieved.
  → Needs improve

In [15]:
# Step 6: Generate report
generate_report(train_count, val_count, train_obj, val_obj, metrics, model_path)
    
print("\n" + "="*70)
print("✓ ALL DONE!")
print("="*70)
    
print("\nNext steps:")
print("  1. Review training plots in output directory")
print("  2. Test on validation patches:")
print(f"     yolo predict model={model_path} source=patches/ conf=0.45")
print("  3. Run visualization script to review predictions")
print("  4. If satisfied, test on new slide")
print("  5. Consider active learning to improve further")
    
print(f"\nModel saved: {model_path}")
print("="*70)


TRAINING SUMMARY REPORT

✓ Report saved: /Users/user/py/Hpylori/yolo_training/training_report.json

📊 Quick Summary:
  Dataset: 232 consensus patches
  Objects: 302 bacteria
  mAP50: 50.3%
  F1: 53.1%
  Model: /Users/user/py/Hpylori/yolo_training/train_20251119_161244/weights/best.pt

✓ ALL DONE!

Next steps:
  1. Review training plots in output directory
  2. Test on validation patches:
     yolo predict model=/Users/user/py/Hpylori/yolo_training/train_20251119_161244/weights/best.pt source=patches/ conf=0.45
  3. Run visualization script to review predictions
  4. If satisfied, test on new slide
  5. Consider active learning to improve further

Model saved: /Users/user/py/Hpylori/yolo_training/train_20251119_161244/weights/best.pt


In [1]:
"""
Advanced YOLO Prediction Visualizer
Features:
- Color-coded confidence levels (Red/Yellow/Orange)
- Side-by-side comparison with ground truth
- Confidence distribution histogram
- Interactive selection of patches
"""

import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# ===== CONFIGURATION =====
MODEL_PATH = "/Users/user/py/Hpylori/yolo_training/train_20251119_161244/weights/best.pt"
IMAGES_DIR = "/Users/user/py/Hpylori/yolo_training/images/val"
LABELS_DIR = "/Users/user/py/Hpylori/yolo_training/labels/val"
OUTPUT_DIR = "/Users/user/py/Hpylori/visualizations"

NUM_SAMPLES = 12  # Number of patches to show
SHOW_GROUND_TRUTH = True  # Show ground truth boxes in green

# Confidence thresholds
HIGH_CONF = 0.5
MEDIUM_CONF = 0.25
DETECTION_CONF = 0.25  # Only show detections >= this confidence (medium and above)

# ===== LOAD MODEL =====
print("="*70)
print("YOLO PREDICTION VISUALIZER")
print("="*70)
print("\nLoading model...")
model = YOLO(MODEL_PATH)
print(f"✓ Loaded: {MODEL_PATH}")

# ===== HELPER FUNCTIONS =====
def get_confidence_color(conf):
    """Returns BGR color and label based on confidence"""
    if conf >= HIGH_CONF:
        return (255, 0, 0), 'HIGH'  # Red
    elif conf >= MEDIUM_CONF:
        return (255, 255, 0), 'MED'  # Yellow
    else:
        return (0, 165, 255), 'LOW'  # Orange

def load_ground_truth(label_path, img_shape):
    """Load YOLO format ground truth boxes"""
    boxes = []
    if not label_path.exists():
        return boxes
    
    h, w = img_shape[:2]
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                # YOLO format: class x_center y_center width height (normalized)
                cls, x_c, y_c, bw, bh = map(float, parts[:5])
                
                # Convert to pixel coordinates
                x1 = int((x_c - bw/2) * w)
                y1 = int((y_c - bh/2) * h)
                x2 = int((x_c + bw/2) * w)
                y2 = int((y_c + bh/2) * h)
                
                boxes.append((x1, y1, x2, y2))
    return boxes

def draw_detections(img, predictions, ground_truth=None):
    """Draw predictions and optionally ground truth on image"""
    img_annotated = img.copy()
    
    # Draw ground truth in green (if available)
    if ground_truth is not None and SHOW_GROUND_TRUTH:
        for x1, y1, x2, y2 in ground_truth:
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img_annotated, 'GT', (x1, y1-5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Draw predictions with color-coded confidence
    for det in predictions:
        x1, y1, x2, y2 = det['coords']
        conf = det['conf']
        color, label = get_confidence_color(conf)
        
        # Draw box
        cv2.rectangle(img_annotated, (x1, y1), (x2, y2), color, 2)
        
        # Draw label with background
        text = f"{label} {conf:.2f}"
        (w, h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img_annotated, (x1, y1-h-8), (x1+w, y1), color, -1)
        cv2.putText(img_annotated, text, (x1, y1-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    return img_annotated

# ===== PROCESS PATCHES =====
print("\nScanning for patches...")
images_path = Path(IMAGES_DIR)
labels_path = Path(LABELS_DIR)

all_images = list(images_path.glob('*.png'))
if len(all_images) == 0:
    print(f"❌ No images found in {IMAGES_DIR}")
    exit(1)

print(f"✓ Found {len(all_images)} images")

# Sample random patches
num_samples = min(NUM_SAMPLES, len(all_images))
selected_images = random.sample(all_images, num_samples)

print(f"✓ Selected {num_samples} random samples")
print("\nRunning inference...")

# Process each image
results_data = []
all_confidences = []

for i, img_path in enumerate(selected_images, 1):
    print(f"  [{i}/{num_samples}] {img_path.name}")
    
    # Run YOLO inference
    results = model.predict(
        source=str(img_path),
        conf=DETECTION_CONF,
        iou=0.45,
        verbose=False
    )[0]
    
    # Load image
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Extract predictions
    predictions = []
    if results.boxes is not None and len(results.boxes) > 0:
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            
            predictions.append({
                'coords': (int(x1), int(y1), int(x2), int(y2)),
                'conf': conf
            })
            all_confidences.append(conf)
    
    # Load ground truth
    label_path = labels_path / (img_path.stem + '.txt')
    ground_truth = load_ground_truth(label_path, img.shape)
    
    # Store results
    results_data.append({
        'path': img_path,
        'image': img_rgb,
        'predictions': predictions,
        'ground_truth': ground_truth
    })

# ===== CREATE VISUALIZATIONS =====
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

print("\n" + "="*70)
print("CREATING VISUALIZATIONS")
print("="*70)

# 1. MAIN GRID VISUALIZATION
print("\n1. Creating prediction grid...")
n_cols = 4
n_rows = (num_samples + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for idx, result in enumerate(results_data):
    row = idx // n_cols
    col = idx % n_cols
    ax = axes[row, col]
    
    # Draw annotations
    img_annotated = draw_detections(
        result['image'], 
        result['predictions'],
        result['ground_truth']
    )
    
    ax.imshow(img_annotated)
    
    # Create title
    n_pred = len(result['predictions'])
    n_gt = len(result['ground_truth'])
    high = sum(1 for p in result['predictions'] if p['conf'] >= HIGH_CONF)
    med = sum(1 for p in result['predictions'] 
              if MEDIUM_CONF <= p['conf'] < HIGH_CONF)
    low = sum(1 for p in result['predictions'] if p['conf'] < MEDIUM_CONF)
    
    title = f"{result['path'].name}\n"
    if SHOW_GROUND_TRUTH:
        title += f"GT:{n_gt} | "
    title += f"Pred:{n_pred} (H:{high} M:{med} L:{low})"
    
    ax.set_title(title, fontsize=9)
    ax.axis('off')

# Hide empty subplots
for idx in range(num_samples, n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    axes[row, col].axis('off')

# Add legend
legend_elements = [
    mpatches.Patch(color='red', label=f'HIGH (≥{HIGH_CONF:.2f})'),
    mpatches.Patch(color='yellow', label=f'MED ({MEDIUM_CONF:.2f}-{HIGH_CONF:.2f})'),
    mpatches.Patch(color='orange', label=f'LOW (<{MEDIUM_CONF:.2f})')
]
if SHOW_GROUND_TRUTH:
    legend_elements.append(mpatches.Patch(color='lime', label='Ground Truth'))

fig.legend(handles=legend_elements, loc='upper center',
          bbox_to_anchor=(0.5, 0.98), ncol=4, fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.97])
grid_file = output_path / 'predictions_grid.png'
plt.savefig(grid_file, dpi=150, bbox_inches='tight')
print(f"   ✓ Saved: {grid_file}")

# 2. CONFIDENCE DISTRIBUTION
if all_confidences:
    print("\n2. Creating confidence histogram...")
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(all_confidences, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(HIGH_CONF, color='red', linestyle='--', linewidth=2, 
               label=f'High threshold ({HIGH_CONF})')
    ax.axvline(MEDIUM_CONF, color='orange', linestyle='--', linewidth=2,
               label=f'Medium threshold ({MEDIUM_CONF})')
    
    ax.set_xlabel('Confidence Score', fontsize=12)
    ax.set_ylabel('Number of Detections', fontsize=12)
    ax.set_title('Distribution of Detection Confidence Scores', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    hist_file = output_path / 'confidence_distribution.png'
    plt.savefig(hist_file, dpi=150, bbox_inches='tight')
    print(f"   ✓ Saved: {hist_file}")

# 3. SAVE INDIVIDUAL IMAGES
print("\n3. Saving individual annotated images...")
individual_dir = output_path / 'individual'
individual_dir.mkdir(exist_ok=True)

for result in results_data:
    img_annotated = draw_detections(
        result['image'],
        result['predictions'],
        result['ground_truth']
    )
    
    output_file = individual_dir / f"pred_{result['path'].name}"
    cv2.imwrite(str(output_file), cv2.cvtColor(img_annotated, cv2.COLOR_RGB2BGR))

print(f"   ✓ Saved {len(results_data)} images to {individual_dir}")

# ===== PRINT STATISTICS =====
print("\n" + "="*70)
print("DETECTION STATISTICS")
print("="*70)

total_pred = sum(len(r['predictions']) for r in results_data)
total_gt = sum(len(r['ground_truth']) for r in results_data)
high = sum(sum(1 for p in r['predictions'] if p['conf'] >= HIGH_CONF) 
          for r in results_data)
med = sum(sum(1 for p in r['predictions'] 
              if MEDIUM_CONF <= p['conf'] < HIGH_CONF) 
         for r in results_data)
low = sum(sum(1 for p in r['predictions'] if p['conf'] < MEDIUM_CONF) 
         for r in results_data)

print(f"\nPatches analyzed: {num_samples}")
if SHOW_GROUND_TRUTH:
    print(f"Ground truth bacteria: {total_gt}")
print(f"Predicted bacteria: {total_pred}")

print(f"\nConfidence breakdown:")
print(f"  HIGH  (≥{HIGH_CONF:.2f}): {high:3d} ({high/max(total_pred,1)*100:5.1f}%)")
print(f"  MED   ({MEDIUM_CONF:.2f}-{HIGH_CONF:.2f}): {med:3d} ({med/max(total_pred,1)*100:5.1f}%)")
print(f"  LOW   (<{MEDIUM_CONF:.2f}): {low:3d} ({low/max(total_pred,1)*100:5.1f}%)")

if all_confidences:
    print(f"\nConfidence statistics:")
    print(f"  Mean: {np.mean(all_confidences):.3f}")
    print(f"  Median: {np.median(all_confidences):.3f}")
    print(f"  Min: {np.min(all_confidences):.3f}")
    print(f"  Max: {np.max(all_confidences):.3f}")

print("\n" + "="*70)
print("✓ VISUALIZATION COMPLETE")
print("="*70)
print(f"\nOutput directory: {output_path}")
print(f"  • predictions_grid.png - Main visualization")
print(f"  • confidence_distribution.png - Histogram")
print(f"  • individual/ - Individual annotated images")
print("\nColor legend:")
print("  🔴 RED    = High confidence (≥0.50)")
print("  🟡 YELLOW = Medium confidence (0.25-0.50)")
print("  🟠 ORANGE = Low confidence (<0.25)")
if SHOW_GROUND_TRUTH:
    print("  🟢 GREEN  = Ground truth")

plt.show()

YOLO PREDICTION VISUALIZER

Loading model...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/user/py/Hpylori/yolo_training/train_20251119_161244/weights/best.pt'